# 26-Class ECG Classifier V3.2 (PTB-XL + Chapman + Challenge + PTB + CODE-15%)

Trains ECGNetJoint on **~458K records** with transfer learning from V3 best (AUROC=0.9682).

---
| Cell | Runtime | Time | What |
|------|---------|------|------|
| 1 | **GPU/TPU** | ~60-75 min | Verify 5 datasets + setup |
| 2 | **GPU/TPU** | ~40-90 min (TPU) | Train V3.2 + save model to Drive |

**Before running:** extract all datasets locally to `/ekg_datasets/` and wait for Drive to sync.

**Datasets (all pre-extracted, no tar extraction needed):**
- **PTB-XL** (~2.7 GB, 21,799 .dat files, 18,524 records)
- **Chapman** (~5.5 GB, 45,152 .mat files, 42,390 records)
- **Challenge 2021** (~7 GB, 54,346 .mat files, 50,842 records)
- **PTB** (PhysioNet, 394 .mat files, ~500 records) â€” *optional*
- **CODE-15%** (18 HDF5 parts, ~345K records) â€” *download parts 0-11 (~42 GB) or all 18 (~60 GB)*

**Total records:** PTB-XL (18.5K) + Chapman (42.4K) + Challenge (50.8K) + PTB (0.5K) + CODE-15% (345K) = **~457K**


In [5]:
# Cell 1 -- GPU/TPU runtime | Setup + verify pre-extracted datasets
#
# DATASETS EXPECTED (synced from local PC via Google Drive Desktop):
#   Drive: MyDrive/EKG/ekg_datasets/ptbxl/         (~2.6 GB) -> copied to SSD
#   Drive: MyDrive/EKG/ekg_datasets/chapman/        (~5.4 GB) -> copied to SSD
#   Drive: MyDrive/EKG/ekg_datasets/challenge2021/  (~7.1 GB) -> copied to SSD
#   Drive: MyDrive/EKG/ekg_datasets/ptb/            (~1.0 GB) -> copied to SSD
#   Drive: MyDrive/EKG/ekg_datasets/code15/         (~64 GB)  -> copied to SSD
#
# STORAGE STRATEGY:
#   All datasets copied to SSD for fast random I/O during training.
#   Small datasets: ~16 GB.  CODE-15%: ~64 GB.  Total: ~80 GB.
#   Colab plan sizes: TPU v6e ~225-256 GB, GPU Pro ~150 GB, free ~112 GB.
#   Code15 copy is capped dynamically by (avail - 4 GB reserve) / 3.6 GB per part.
#   SSD HDF5 reads are 10-25x faster than Drive -- critical for CODE-15% throughput.
#
# CODE-15% INDEX: automatically rebuilt if stale or missing.
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -------

import time, os, sys, subprocess, shutil, tarfile
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 1: GPU/TPU setup + restore data (dynamic-sized SSD plan)')
print('=' * 60)

# ---- 1. Accelerator check ----
import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU found! Switch to GPU/TPU runtime.')
print(f'Accelerator: {accel}')

# ---- 2. Mount Drive ----
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE_ROOT = Path('/content/drive/MyDrive/EKG')
DRIVE_DATASETS = DRIVE_ROOT / 'ekg_datasets'
SSD_EKG = Path('/content/ekg_datasets')
print('Mounted at /content/drive')

# ---- 3. SSD space check ----
stat = os.statvfs('/content')
free_gb = stat.f_frsize * stat.f_bavail / 1e9
need_gb = 20  # minimum for small datasets (~16 GB); code15 copies if space allows
print(f'SSD free: {free_gb:.0f} GB')
if free_gb < need_gb:
    raise RuntimeError(
        f'Only {free_gb:.0f} GB SSD free, need at least {need_gb} GB. Restart runtime.'
    )

# ---- 4. Install dependencies ----
print('Installing deps...', flush=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'wfdb', 'scipy', 'scikit-learn', 'h5py'],
    check=True
)
# Free pip cache to maximize SSD for datasets
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'],
               capture_output=True, check=False)
print(f'Deps ready ({time.time()-t0:.0f}s)')

# ---- 5. Copy training scripts from Drive ----
SCRIPTS = [
    'multilabel_v3.py', 'multilabel_classifier.py', 'cnn_classifier.py',
    'dataset_chapman.py', 'dataset_challenge.py', 'dataset_code15.py', 'dataset_ptb.py',
]
missing_scripts = []
for s in SCRIPTS:
    src = DRIVE_ROOT / s
    if src.exists():
        shutil.copy(str(src), f'/content/{s}')
    else:
        missing_scripts.append(s)
if missing_scripts:
    print(f'WARNING: scripts not on Drive: {missing_scripts}')
print(f'Scripts copied ({time.time()-t0:.0f}s)')

# ---- 6. Copy datasets Drive -> SSD ----
SSD_EKG.mkdir(parents=True, exist_ok=True)

SMALL_DATASETS = [
    ('ptbxl',        'ptbxl',         True,  2.6),
    ('chapman',      'chapman',        True,  5.4),
    ('challenge2021','challenge2021',  True,  7.1),
    ('ptb',          'ptb',            False, 1.0),
]
# Small datasets: FULL COPY to SSD (not symlink).
# Reading 120K+ .mat/.dat files through Drive FUSE during preload_signals()
# was taking 5+ hours. A one-time copy (~16 GB, ~15 min) avoids that bottleneck
# for every subsequent run within this runtime. CODE-15% HDF5 parts are also
# copied to SSD (random access per batch, too large to preload into RAM).
def _copy_with_progress(src_ds, dst, label):
    '''Copy directory tree file-by-file with heartbeat prints (~every 2000 files).
    Robust to interrupted prior runs: idempotent per-file, skips already-copied files.
    '''
    ct0 = time.time()
    all_files = [p for p in src_ds.rglob('*') if p.is_file()]
    total = len(all_files)
    print(f'  {label}: {total:,} files to copy from Drive', flush=True)
    dst.mkdir(parents=True, exist_ok=True)
    copied = skipped = 0
    for i, sf in enumerate(all_files):
        rel = sf.relative_to(src_ds)
        df = dst / rel
        if df.exists() and df.stat().st_size == sf.stat().st_size:
            skipped += 1
        else:
            df.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(sf), str(df))
            copied += 1
        if (i + 1) % 2000 == 0 or i == total - 1:
            elapsed = time.time() - ct0
            rate = (i + 1) / max(elapsed, 0.1)
            eta = (total - i - 1) / max(rate, 0.1)
            print(f'    {i+1:,}/{total:,} ({rate:.0f} files/s, ETA {eta:.0f}s)', flush=True)
    print(f'  {label}: done ({time.time()-ct0:.0f}s, copied={copied:,} skipped={skipped:,})', flush=True)
    return total

for ssd_name, drive_name, required, approx_gb in SMALL_DATASETS:
    dst = SSD_EKG / ssd_name
    src_ds = DRIVE_DATASETS / drive_name
    if dst.is_symlink():
        # Stale symlink from a previous run — remove so we can copy fresh
        dst.unlink()
        print(f'{ssd_name}: removed stale Drive symlink')
    # Partial-copy detection: compare dst file count to src file count.
    # If equal, skip; if less, resume (copy_with_progress is idempotent).
    if dst.exists() and not dst.is_symlink():
        # Self-heal double-nested extraction from a prior run
        nested = dst / drive_name
        if nested.is_dir():
            marker_files = {
                'ptbxl':        'ptbxl_database.csv',
                'chapman':      'ConditionNames_SNOMED-CT.csv',
                'challenge2021':'georgia',
                'ptb':          'RECORDS',
            }
            marker = marker_files.get(drive_name)
            # Flatten only if the marker is inside the nested folder (confirms bug)
            if marker is None or (nested / marker).exists() or any((nested / marker).rglob('*') for _ in [0] if marker):
                print(f'{ssd_name}: flattening double-nested {drive_name}/{drive_name}/...', flush=True)
                for item in list(nested.iterdir()):
                    target = dst / item.name
                    if target.exists():
                        continue  # don't clobber anything already at top level
                    shutil.move(str(item), str(target))
                try:
                    nested.rmdir()
                except OSError:
                    pass
        tar_for_check = DRIVE_DATASETS / f'{drive_name}.tar'
        # If a tarball is present we trust it -- wipe any stale partial copy
        # so the fast path below gets a clean extract target.
        if tar_for_check.exists():
            n_dst = sum(1 for _ in dst.rglob('*') if _.is_file())
            # Only wipe if the dst looks incomplete (small) vs the tarball's uncompressed size
            tar_gb = tar_for_check.stat().st_size / 1e9
            dst_bytes = sum(p.stat().st_size for p in dst.rglob('*') if p.is_file())
            if dst_bytes < 0.9 * tar_for_check.stat().st_size:
                print(f'{ssd_name}: partial copy ({dst_bytes/1e9:.1f} GB) found, tarball available — wiping and re-extracting', flush=True)
                shutil.rmtree(str(dst))
            else:
                print(f'{ssd_name}: already on SSD ({n_dst:,} files, {dst_bytes/1e9:.1f} GB)')
                continue
        elif src_ds.exists():
            n_src = sum(1 for _ in src_ds.rglob('*') if _.is_file())
            n_dst = sum(1 for _ in dst.rglob('*') if _.is_file())
            if n_dst >= n_src:
                print(f'{ssd_name}: already on SSD ({n_dst:,} files)')
                continue
            else:
                print(f'{ssd_name}: partial copy on SSD ({n_dst:,}/{n_src:,}) — resuming', flush=True)
        else:
            n_dst = sum(1 for _ in dst.rglob('*') if _.is_file())
            print(f'{ssd_name}: already on SSD ({n_dst:,} files, Drive src missing)')
            continue
    # Tarball fast path: Drive FUSE is ~50x faster on one big file than many small files.
    # User pre-tars on local PC (tools/tar_datasets_for_drive.py), Drive syncs the .tar,
    # Cell 1 copies + extracts locally -- ~3-5 min vs ~3-6 hours.
    tar_path = DRIVE_DATASETS / f'{drive_name}.tar'
    if tar_path.exists():
        sz_gb = tar_path.stat().st_size / 1e9
        print(f'{ssd_name}: found tarball ({sz_gb:.1f} GB), using fast path', flush=True)
        tt0 = time.time()
        local_tar = Path('/content') / f'{drive_name}.tar'
        print(f'  copying {tar_path.name} from Drive...', flush=True)
        shutil.copy2(str(tar_path), str(local_tar))
        print(f'  copied ({time.time()-tt0:.0f}s), extracting...', flush=True)
        ex0 = time.time()
        with tarfile.open(str(local_tar), 'r') as tf:
            # If the archive is wrapped in a single top-level folder named the
            # same as drive_name (e.g. `tar -cf ptbxl.tar ptbxl/`), extract to
            # the parent so files land at SSD_EKG/<drive_name>/... instead of
            # SSD_EKG/<drive_name>/<drive_name>/... (double-nesting bug).
            top_levels = {n.split('/', 1)[0] for n in tf.getnames() if n and not n.startswith('/')}
            if len(top_levels) == 1 and next(iter(top_levels)) == drive_name:
                extract_to = SSD_EKG
                SSD_EKG.mkdir(parents=True, exist_ok=True)
                print(f'  archive wraps in {drive_name}/, extracting to parent', flush=True)
            else:
                extract_to = dst
                dst.mkdir(parents=True, exist_ok=True)
            tf.extractall(str(extract_to))
        print(f'  extracted ({time.time()-ex0:.0f}s)', flush=True)
        local_tar.unlink()
        n_files = sum(1 for _ in dst.rglob('*') if _.is_file())
        print(f'  {ssd_name}: done ({time.time()-tt0:.0f}s, {n_files:,} files)', flush=True)
        continue
    if src_ds.exists():
        print(f'{ssd_name}: copying from Drive (~{approx_gb} GB)...', flush=True)
        _copy_with_progress(src_ds, dst, ssd_name)
    elif required:
        raise RuntimeError(
            f'Dataset "{ssd_name}" not found at {src_ds}. '
            f'Sync ekg_datasets/{drive_name}/ from local PC via Google Drive Desktop.'
        )
    else:
        print(f'{ssd_name}: not found on Drive (optional, skipping)')

# code15: copy as many parts as fit on SSD (no Drive FUSE during training)
code15_src = DRIVE_DATASETS / 'code15'
code15_dst = SSD_EKG / 'code15'
if not code15_src.exists():
    raise RuntimeError(
        'CODE-15% not found at ' + str(code15_src) + '. '
        'Sync ekg_datasets/code15/ from local PC via Google Drive Desktop.'
    )
# Remove stale symlink if present from a previous run
if code15_dst.is_symlink():
    code15_dst.unlink()
    print('code15: removed old Drive symlink')
code15_dst.mkdir(parents=True, exist_ok=True)
(code15_dst / 'raw').mkdir(exist_ok=True)
# Copy exams.csv (small metadata file)
exams_csv_src = code15_src / 'raw' / 'exams.csv'
if exams_csv_src.exists() and not (code15_dst / 'raw' / 'exams.csv').exists():
    shutil.copy2(str(exams_csv_src), str(code15_dst / 'raw' / 'exams.csv'))
# Calculate how many HDF5 parts fit (leave 4 GB reserve for checkpoints)
drive_parts = sorted((code15_src / 'raw').glob('exams_part*.hdf5'))
n_drive_parts = len(drive_parts)
stat_now = os.statvfs('/content')
avail_gb = stat_now.f_frsize * stat_now.f_bavail / 1e9
reserve_gb = 4
per_part_gb = 3.6
max_parts = min(n_drive_parts, max(0, int((avail_gb - reserve_gb) / per_part_gb)))
print(f'SSD available: {avail_gb:.0f} GB -> will copy {max_parts}/{n_drive_parts} code15 parts')
if max_parts == 0:
    raise RuntimeError('Not enough SSD space for any code15 part. Need ' + str(reserve_gb + per_part_gb) + '+ GB free.')
for part_file in drive_parts[:max_parts]:
    dst_file = code15_dst / 'raw' / part_file.name
    if dst_file.exists() and dst_file.stat().st_size > 100_000:
        print(f'  {part_file.name}: already on SSD', flush=True)
    else:
        sz = part_file.stat().st_size / 1e9
        print(f'  Copying {part_file.name} ({sz:.1f} GB)...', flush=True)
        shutil.copy2(str(part_file), str(dst_file))
        print(f'    done ({time.time()-t0:.0f}s)', flush=True)
n_ssd_parts = len(list((code15_dst / 'raw').glob('exams_part*.hdf5')))
print(f'code15: {n_ssd_parts}/{n_drive_parts} parts on SSD ({time.time()-t0:.0f}s)')

# Copy index CSV files (needed by dataset_chapman.py and multilabel_v3.py)
for csv_name in ['chapman_index.csv', 'unified_index.csv']:
    dst_csv = SSD_EKG / csv_name
    src_csv = DRIVE_DATASETS / csv_name
    if dst_csv.exists():
        print(f'{csv_name}: already on SSD')
    elif src_csv.exists():
        shutil.copy2(str(src_csv), str(dst_csv))
        print(f'{csv_name}: copied from Drive')
    else:
        print(f'WARNING: {csv_name} not on Drive')

# If chapman_index.csv still missing, rebuild from Chapman data on SSD
if not (SSD_EKG / 'chapman_index.csv').exists() and (SSD_EKG / 'chapman').exists():
    print('Rebuilding chapman_index.csv (~1 min)...', flush=True)
    res = subprocess.run(
        [sys.executable, '/content/dataset_chapman.py', '--index'],
        capture_output=True, text=True, cwd='/content'
    )
    if res.returncode == 0:
        print(f'chapman_index.csv built ({time.time()-t0:.0f}s)')
    else:
        raise RuntimeError('Failed to build chapman index: ' + res.stderr[-500:])

print(f'All datasets ready ({time.time()-t0:.0f}s)')

# ---- 7. Verify datasets ----
print('\nVerifying datasets...')
errors = []

ptbxl_csv = SSD_EKG / 'ptbxl' / 'ptbxl_database.csv'
if ptbxl_csv.exists():
    print('  PTB-XL: OK')
else:
    errors.append('PTB-XL: ptbxl_database.csv missing')

for name, pattern, min_count in [('chapman', '*.mat', 44000), ('challenge2021', '*.mat', 50000)]:
    d = SSD_EKG / name
    n = sum(1 for _ in d.rglob(pattern)) if d.exists() else 0
    if n >= min_count:
        print(f'  {name}: OK ({n:,} files)')
    elif n > 0:
        print(f'  {name}: WARNING {n:,} files (expected {min_count:,}+, continuing)')
    else:
        errors.append(f'{name}: directory missing or empty')

code15_raw = SSD_EKG / 'code15' / 'raw'
if code15_raw.exists():
    n_parts = len(sorted(code15_raw.glob('exams_part*.hdf5')))
    label = 'all 18/18' if n_parts == 18 else f'{n_parts}/18'
    print(f'  CODE-15%: {label} parts (~{n_parts * 19000:,} records) on SSD')
    if n_parts == 0:
        errors.append('CODE-15%: no HDF5 parts found')
    elif n_parts < 18:
        print(f'  CODE-15%: WARNING -- only {n_parts}/18 parts. Full run needs all 18.')
else:
    errors.append(f'CODE-15%: code15/raw/ not found at {code15_raw}')

if errors:
    print('\nERRORS:')
    for e in errors: print(f'  x {e}')
    raise RuntimeError('Dataset verification failed.')
print(f'All datasets verified OK ({time.time()-t0:.0f}s)')

# ---- 8. Rebuild CODE-15% index if stale or missing ----
code15_index = SSD_EKG / 'code15' / 'code15_index.csv'
if code15_raw.exists():
    n_parts = len(sorted(code15_raw.glob('exams_part*.hdf5')))
    if n_parts > 0:
        needs_build = not code15_index.exists()
        if code15_index.exists():
            idx_mtime = code15_index.stat().st_mtime
            newest_part = max(p.stat().st_mtime for p in code15_raw.glob('exams_part*.hdf5'))
            if newest_part > idx_mtime:
                print('CODE-15% index stale -- rebuilding.')
                needs_build = True
        if needs_build:
            print(f'Building CODE-15% index ({n_parts} parts, ~5 min)...', flush=True)
            res = subprocess.run(
                [sys.executable, '/content/dataset_code15.py', '--index'],
                capture_output=True, text=True
            )
            if res.returncode == 0:
                print(f'CODE-15% index built ({time.time()-t0:.0f}s)')
            else:
                print(f'WARNING: index build failed:\n{res.stderr[-500:]}')
        else:
            print(f'CODE-15% index: up to date ({n_parts} parts).')

print(f'\nCell 1 done ({time.time()-t0:.0f}s)')
stat2 = os.statvfs('/content')
used_gb = (stat.f_blocks - stat2.f_bavail) * stat.f_frsize / 1e9
print(f'SSD used: ~{used_gb:.0f} GB (~80 GB for all datasets on SSD)')
print('Ready to train. Run Cell 2.')

Cell 1: GPU/TPU setup + restore data (dynamic-sized SSD plan)
Accelerator: TPU
Mounted at /content/drive
Mounted at /content/drive
SSD free: 112 GB
Installing deps...
Deps ready (4s)
Scripts copied (5s)
ptbxl: already on SSD (43,602 files, 2.6 GB)
chapman: already on SSD (90,757 files, 5.5 GB)
challenge2021: already on SSD (108,679 files, 7.2 GB)
ptb: already on SSD (743 files)
SSD available: 112 GB -> will copy 18/18 code15 parts
  exams_part0.hdf5: already on SSD
  exams_part1.hdf5: already on SSD
  exams_part10.hdf5: already on SSD
  exams_part11.hdf5: already on SSD
  exams_part12.hdf5: already on SSD
  exams_part13.hdf5: already on SSD
  exams_part14.hdf5: already on SSD
  exams_part15.hdf5: already on SSD
  exams_part16.hdf5: already on SSD
  exams_part17.hdf5: already on SSD
  exams_part2.hdf5: already on SSD
  exams_part3.hdf5: already on SSD
  exams_part4.hdf5: already on SSD
  exams_part5.hdf5: already on SSD
  exams_part6.hdf5: already on SSD
  exams_part7.hdf5: already on S

In [6]:
# -----------------------------------------------------------------------------
# Cell 2 -- GPU/TPU runtime | Train V3.2 + save to Drive
# TPU (v6e): ~40-90 min  |  GPU (T4): ~2-4 hrs
# First TPU epoch is slow (XLA compilation) -- this is normal.
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil, re
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 2: Train V3.2 + save model')
print('=' * 60)
os.chdir('/content')

import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Run Cell 1 first.')

if accel == 'TPU':
    try:
        fd = os.open('/dev/vfio/0', os.O_RDONLY | os.O_NONBLOCK)
        os.close(fd)
    except OSError as e:
        raise RuntimeError(
            f'TPU device busy ({e}). Runtime > Restart runtime, then re-run Cell 1 + 2.'
        )

print(f'Accelerator: {accel}')

# Create models/ dir so torch.save() can write checkpoints
Path('/content/models').mkdir(parents=True, exist_ok=True)
print('Created /content/models/')

print('Starting multilabel_v3.py...', flush=True)
print('-' * 60, flush=True)

proc = subprocess.Popen(
    [sys.executable, '-u', 'multilabel_v3.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
_captured = []
for line in proc.stdout:
    print(line, end='', flush=True)
    _captured.append(line.rstrip())
proc.wait()

print('-' * 60, flush=True)
if proc.returncode != 0:
    raise RuntimeError(f'Training failed (exit {proc.returncode})')

src = Path('/content/models/ecg_multilabel_v3.pt')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
if src.exists():
    dst = DRIVE_MODELS / 'ecg_multilabel_v3.pt'
    shutil.copy(src, dst)
    print(f'Model saved to Drive ({dst.stat().st_size/1e6:.0f} MB)')
    print('Syncs to local PC via Google Drive for Desktop.')
else:
    print('WARNING: model file not found')

# Copy best checkpoint too if it exists
src_best = Path('/content/models/ecg_multilabel_v3_best.pt')
if src_best.exists():
    dst_best = DRIVE_MODELS / 'ecg_multilabel_v3_best.pt'
    shutil.copy(src_best, dst_best)
    print(f'Best model saved to Drive ({dst_best.stat().st_size/1e6:.0f} MB)')

print(f'\nCell 2 done ({time.time()-t0:.0f}s)')

Cell 2: Train V3.2 + save model
Accelerator: TPU
Created /content/models/
Starting multilabel_v3.py...
------------------------------------------------------------

  V3 Multi-Label ECG Classifier  (26 conditions)
  PTB-XL + Chapman + PhysioNet 2021 Challenge + CODE-15%
  Device: TPU (xla:0)
  Batch size: 256
  Learning rate: 1.2e-03 (scaled from 3.0e-04)
  Indexing 21799 records for multi-label...
  Kept 18524 records  (skipped: 3275 no-label, 0 no-file)
  Per-class positives:  NORM=9438  PVC=1027  LVH=1751  IMI=1714  ASMI=2007  CLBBB=536  CRBBB=540  LAFB=1622  1AVB=790  ISC_=1260  NDT=1824  IRBBB=1115
Chapman-Shaoxing: 42390 records loaded
Loading Challenge datasets...
  [challenge] georgia         :  10045 kept  (skipped: 238 no-label, 34 no-hea)
  [challenge] cpsc_2018       :   5634 kept  (skipped: 153 no-label, 46 no-hea)
  [challenge] cpsc_2018_extra :   2461 kept  (skipped: 925 no-label, 27 no-hea)
  [challenge] ningbo          :  32702 kept  (skipped: 1969 no-label, 112 no-hea